# Pendulum swing-up — value iteration vs LQR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/courses/udes_gro860/pendulum_value_iteration_vs_lqr.ipynb)

This notebook compares two ways to synthesize a policy for the **same** swing-up problem and the **same** quadratic cost $J$. Both return a state-feedback law $u=\pi(x)$; they differ in what they assume about the dynamics.

1. **Value iteration (VI)**: global dynamic programming on a grid — the discretized nonlinear optimum, with torque limits.
2. **LQR**: local linear-quadratic feedback from the linearized dynamics at the upright equilibrium.

Same plant, same $Q$ and $R$. The lesson is local vs global synthesis: LQR is cheap and exact near $\bar x$; VI can swing up from the hanging position.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox.


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")

In [ ]:
import numpy as np

from minilink import (
    DynamicProgrammingPlanner,
    LQRPlanner,
    Pendulum,
    PlanningProblem,
    compare,
)

## 1. Plant

We load a minilink catalog class (`Pendulum`) that already defines the equations of motion and the state/input labels. The state is $x = [\theta,\;\dot\theta]$ and the input is the pivot torque $u$. The dynamics are
$$\dot x = f(x,u).$$
Hanging down is $\theta = 0$; the upright target is $\bar x = [-\pi,\; 0]$. We also set the **domain** — bounds on $x$ and $|u|\le u_{\max}$ — used later by the grid.


In [ ]:
UPRIGHT = np.array([-np.pi, 0.0])  # target (upright) and LQR linearization point
X0 = np.array([0.0, 0.0])  # hanging down
TORQUE = 5.0
DT = 0.05
TF = 10.0
X_GRID = (201, 201)
U_GRID = (21,)
TOL = 0.1
INF = 500.0
Q = np.diag([1.0, 1.0])
R = np.diag([1.0])

plant = Pendulum()
plant.state.lower_bound = np.array([-2.0 * np.pi, -2.0 * np.pi])
plant.state.upper_bound = np.array([+2.0 * np.pi, +2.0 * np.pi])
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([+TORQUE])
plant.x0 = X0

## 2. Cost function

Both controllers minimize the same quadratic performance metric
$$J = \int_0^{t_f} \big( (x-\bar x)' Q (x-\bar x) + u' R u \big)\, dt$$
with $Q = I$ and $R = I$. This $J$ is what we compare later on the closed-loop trajectories.


In [ ]:
from minilink import QuadraticCost

cost = QuadraticCost.from_system(plant, xbar=UPRIGHT, Q=Q, R=R)

## 3. Planning problem

A `PlanningProblem` packages the plant, the cost, and the goal. The optimal-control problem is
$$\min_{\pi}\; J \quad\text{s.t.}\quad \dot x = f\big(x,\pi(x)\big),\quad |u|\le u_{\max}.$$
Value iteration solves this on a grid. LQR solves a local linear-quadratic approximation of the same $J$.


In [ ]:
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)

## 4. Value iteration

We discretize $x$ and $u$ on a grid and solve the discrete Bellman equation for the cost-to-go $J^*$:
$$J^*(x) = \min_u \Big\{ g(x,u)\,\Delta t + J^*\big(x + f(x,u)\,\Delta t\big) \Big\}.$$
The minimizing $u$ is the global (discretized) policy $\pi^*(x)$. Here the state grid is $201\times 201$, the torque has 21 levels, and $\Delta t = 0.05\,\mathrm{s}$.


In [ ]:
planner = DynamicProgrammingPlanner(
    problem,
    x_grid=X_GRID,
    u_grid=U_GRID,
    dt=DT,
    tol=TOL,
    max_iterations=2000,
    out_of_bound_cost=INF,
    verbose=True,
)

sol_vi = planner.solve()
vi_ctl = sol_vi.policy

## 5. LQR

Linearize the plant at the upright equilibrium $(\bar x, \bar u)$:
$$\dot{\tilde x} = A\tilde x + B\tilde u,\qquad \tilde x = x-\bar x.$$
The infinite-horizon LQR gain $K$ minimizes the same quadratic $J$ for this linear model, giving the local law
$$u = \bar u - K(x-\bar x).$$
Same $Q$, $R$, and plant as value iteration — but no torque limits in the synthesis, and no validity away from $\bar x$. `LQRPlanner` linearizes the plant at the cost's target and solves the Riccati equation; its solution carries the law, the closed-loop poles, and the cost-to-go of the linear model.

In [ ]:
sol_lqr = LQRPlanner(problem).solve()
lqr_ctl = sol_lqr.policy
print(sol_lqr.solver)

## 6. Control laws

Both maps $u=\pi(\theta,\dot\theta)$ on the same state box and the same torque scale. LQR is a linear plane; VI is a nonlinear lookup that follows the natural dynamics. Near $\bar x$ they look similar; globally LQR asks for much larger torques. `compare` names the two solutions; `print` reads their solver records side by side, and each `plot_*` draws both on one figure.

In [ ]:
race = compare(VI=sol_vi, LQR=sol_lqr)
print(race)

In [ ]:
race.plot_control_law()

Each method's own cost-to-go, on one colour scale: the value-iteration table, and the Riccati form $(x-\bar x)^T S (x-\bar x)$ of the linear model, which is exact only where the linear model is.

In [ ]:
race.plot_cost_to_go(jmax=INF)

## 7. Closed-loop simulation

We wire each policy as state feedback $u=\pi(x)$ and integrate from the hanging position $x_0 = [0,\; 0]$. LQR goes straight to the goal with large torque; VI pumps energy, then swings up.


In [ ]:
plant.x0 = X0
n_steps = int(TF / DT) + 1  # same step as the DP discretization

cl_vi = vi_ctl @ plant  # state feedback: plant.x -> ctl.x, ctl.u -> plant.u
cl_vi.name = "Pendulum with VI"
traj_vi = cl_vi.compute_trajectory(tf=TF, n_steps=n_steps, solver="euler")

cl_lqr = lqr_ctl @ plant
cl_lqr.name = "Pendulum with LQR"
traj_lqr = cl_lqr.compute_trajectory(tf=TF, n_steps=n_steps, solver="euler")

In [ ]:
cl_vi.plot_trajectory(traj_vi)

In [ ]:
cl_lqr.plot_trajectory(traj_lqr)

## 8. Animation — VI

Closed-loop motion under the value-iteration policy, from hanging down.


In [ ]:
cl_vi.animate(traj_vi)

## 8. Animation — LQR

Same initial state under LQR. Compare the torque and the path to the VI animation.


In [ ]:
cl_lqr.animate(traj_lqr)

## 9. Phase plane

Trajectories in the $(\theta,\dot\theta)$ plane. The vector field is the unactuated dynamics $\dot x = f(x,0)$. VI rides those orbits; LQR cuts across them.


In [ ]:
plant.plot_phase_plane(traj_vi)

In [ ]:
plant.plot_phase_plane(traj_lqr)

## 10. Performance

The same $J$ evaluated along each closed-loop trajectory: the running cost and its integral, then the total. VI is the global optimum of the discretized problem, so its $J$ should be lower from the hanging position. LQR typically spends more torque (and more cost) getting there.


In [ ]:
cl_vi.plot_cost(cost, of=plant, traj=traj_vi)

In [ ]:
cl_lqr.plot_cost(cost, of=plant, traj=traj_lqr)